In [ ]:
from astropy.table import Table
import numpy as np

t1 = Table.read(r'C:\Users\zeng\Desktop\frb_flux_erd\catalog\catalog\chimefrbcat1.fits')
t2 = Table.read(r'C:\Users\zeng\Desktop\frb_flux_erd\catalog\catalog\chimefrbcat2.fits')

# We need columns: FRBname gl gb S Seu Sel W Weu Wel F Feu Fel DM DM_NE2001 DM_YMW16 SURVEY SN Gain Tsys BW Npol SN0 Ref
# The original code expects these specific columns, but we can write a generic parsing.

fout = open(r'C:\Users\zeng\Desktop\frb_flux_erd\chime_frb_cat.txt', 'w')
fout.write(f'{"FRBname":<12} {"gl":<10} {"gb":<10} {"S":<8} {"Seu":<8} {"Sel":<8} {"W":<8} {"Weu":<8} {"Wel":<8} {"F":<8} {"Feu":<8} {"Fel":<8} {"DM":<10} {"DM_NE2001":<12} {"DM_YMW16":<12} {"SURVEY":<10} {"SN":<8} {"Gain":<8} {"Tsys":<8} {"BW":<10} {"Npol":<8} {"SN0":<8} {"Ref":<15}\n')

def process_table(t, survey_name):
    # filter NaNs in flux or width or DM
    mask = (~np.isnan(t['flux'])) & (~np.isnan(t['bc_width'])) & (~np.isnan(t['dm_fitb'])) & (~np.isnan(t['dm_exc_ne2001']))
    t = t[mask]
    for row in t:
        name = row['tns_name'].replace('FRB', '') if 'tns_name' in t.colnames and 'FRB' in row['tns_name'] else 'FRB'
        gl = row['gl']
        gb = row['gb']
        S = row['flux']
        Seu = row['flux_err'] if 'flux_err' in t.colnames else 0.0
        Sel = row['flux_err'] if 'flux_err' in t.colnames else 0.0
        
        W = row['bc_width'] * 1e3 # s -> ms
        Weu = 0.0
        Wel = 0.0
        
        F = row['fluence']
        Feu = row['fluence_err'] if 'fluence_err' in t.colnames else 0.0
        Fel = row['fluence_err'] if 'fluence_err' in t.colnames else 0.0
        
        DM = row['dm_fitb']
        DM_NE2001 = row['dm_exc_ne2001']
        DM_YMW16 = row['dm_exc_ymw16']
        
        SURVEY = survey_name
        SN = row['bonsai_snr']
        # Placeholders for CHIME:
        Gain = 1.4
        Tsys = 50.0
        BW = 400.0
        Npol = 2
        SN0 = 9.0
        Ref = 'CHIME'
        
        fout.write(f'{name:<12} {gl:<10.3f} {gb:<10.3f} {S:<8.3f} {Seu:<8.3f} {Sel:<8.3f} {W:<8.3f} {Weu:<8.3f} {Wel:<8.3f} {F:<8.3f} {Feu:<8.3f} {Fel:<8.3f} {DM:<10.3f} {DM_NE2001:<12.3f} {DM_YMW16:<12.3f} {SURVEY:<10} {SN:<8.3f} {Gain:<8.1f} {Tsys:<8.1f} {BW:<10.1f} {Npol:<8} {SN0:<8.1f} {Ref:<15}\n')

process_table(t1, 'CHIME_1')
process_table(t2, 'CHIME_2')
fout.close()

# Also write chime_tel_svy.txt
with open(r'C:\Users\zeng\Desktop\frb_flux_erd\chime_tel_svy.txt', 'w') as f:
    f.write('SURVEY     FOV        TIME       Gain    Tsys     Npol     SN0    BW\n')
    f.write('CHIME_1    200        8760       1.4     50       2        9.0     400\n')
    f.write('CHIME_2    200        17520      1.4     50       2        9.0     400\n')

print("Conversion complete!")
